# Embedding Explorer

Use this notebook to sanity-check the project premise on one or more tracks: extract frame-level embeddings with a chosen encoder, inspect self-similarity and MSPF, view PCA components over time, and color frames by cluster assignment.

If you provide multiple tracks, plots are arranged side by side using a shared embedding transform wherever that makes comparison meaningful.

## 1. Configuration

Set `AUDIO_PATHS`, choose an encoder, and set `FRAME_RATE_HZ`. With `POOL_WITHIN_WINDOW=True`, each frame is one pooled embedding over a 10-second encoder window, sampled every `1 / FRAME_RATE_HZ` seconds.

In [ ]:
from pathlib import Path

# One or more audio files. Multiple tracks are plotted next to each other.
AUDIO_PATHS = [
    "/data/EECS-Pauwels-C4DM/mtg-jamendo-raw/mtg-jamendo-dataset/mp3/69/1365069.mp3",
]

# Supported by this notebook: "muq", "mert", "matpac", "music2latent", "usad", "clap".
ENCODER_NAME = "mert"

# Output frame rate when POOL_WITHIN_WINDOW=True.
# 0.1 Hz = one embedding every 10 seconds; 1 Hz = every second; 10 Hz = every 100 ms.
FRAME_RATE_HZ = 0.1

# Pool sequence encoders into one vector per window. This makes FRAME_RATE_HZ exact.
# If False, native per-window sequences are flattened, and the effective frame rate is encoder-specific.
POOL_WITHIN_WINDOW = True

# Optional checkpoint required for MatPAC.
MATPAC_CHECKPOINT = "/data/home/acw749/Dyno/pretrained/matpac_10_2048.pt"

# Runtime controls.
DEVICE = "cuda:0"  # use "cpu" if needed
BATCH_WINDOWS = 16
MAX_BATCH_WINDOWS = 32
LIMIT_SECONDS = None  # e.g. 180 for a quick first look

# MSPF controls. `MSPF_WINDOW_FRAMES` should be small enough for quick solves.
MSPF_WINDOW_FRAMES = 30
MSPF_SIGMA = 10.0
MSPF_LAMBDA = 1e-3

# PCA and clustering controls.
N_PCA_COMPONENTS = 3
N_CLUSTERS = 8
RANDOM_STATE = 42

# Plot controls.
MAX_FRAMES_FOR_SSM = 600  # downsample only for SSM/MSPF plotting if a track is very long
FIGSIZE_PER_TRACK = 4.8


## 2. Imports And Helpers

In [ ]:
import importlib
import math
import sys
from dataclasses import dataclass
from typing import Any

import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dyno").exists():
    PROJECT_ROOT = Path("/data/home/acw749/Dyno")
sys.path.insert(0, str(PROJECT_ROOT))

from dyno.evaluation.temporal import compute_mspf, compute_ssm
torch.set_grad_enabled(False)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

@dataclass
class EncoderSpec:
    target: str
    sample_rate: int
    window_seconds: float
    kwargs: dict[str, Any]


def build_encoder_specs(matpac_checkpoint: str) -> dict[str, EncoderSpec]:
    return {
        "muq": EncoderSpec(
            target="dyno.models.encoders.muq.MuQEncoder",
            sample_rate=24000,
            window_seconds=10.0,
            kwargs={"model_name": "OpenMuQ/MuQ-MuLan-large", "freeze": True},
        ),
        "mert": EncoderSpec(
            target="dyno.models.encoders.mert.MERTEncoder",
            sample_rate=24000,
            window_seconds=10.0,
            kwargs={"model_name": "m-a-p/MERT-v1-330M", "layer_index": -1, "freeze": True},
        ),
        "matpac": EncoderSpec(
            target="dyno.models.encoders.matpac.MatPacEncoder",
            sample_rate=16000,
            window_seconds=10.0,
            kwargs={"checkpoint_path": matpac_checkpoint, "freeze": True},
        ),
        "music2latent": EncoderSpec(
            target="dyno.models.encoders.music2latent.Music2LatentEncoder",
            sample_rate=44100,
            window_seconds=10.0,
            kwargs={"sample_rate": 44100, "freeze": True},
        ),
        "usad": EncoderSpec(
            target="dyno.models.encoders.usad.USADEncoder",
            sample_rate=16000,
            window_seconds=10.0,
            kwargs={"model_name": "MIT-SLS/USAD-Large", "freeze": True},
        ),
        "clap": EncoderSpec(
            target="dyno.models.encoders.clap.CLAPEncoder",
            sample_rate=48000,
            window_seconds=10.0,
            kwargs={"model_name": "laion/clap-htsat-fused", "freeze": True},
        ),
    }


In [ ]:
def instantiate_encoder(name: str, pool: bool, device: str):
    specs = build_encoder_specs(MATPAC_CHECKPOINT)
    if name not in specs:
        raise ValueError(f"Unknown encoder '{name}'. Choose one of: {sorted(specs)}")
    spec = specs[name]
    kwargs = dict(spec.kwargs)
    kwargs.setdefault("pool", pool)
    kwargs.setdefault("downsample_factor", 1)
    module_name, class_name = spec.target.rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(module_name), class_name)
    encoder = encoder_cls(**kwargs).eval().to(device)
    if hasattr(encoder, "pool"):
        encoder.pool = pool
    return encoder, spec


def load_audio_windows(path: str, sample_rate: int, window_seconds: float, frame_rate_hz: float, limit_seconds: float | None):
    y, _ = librosa.load(path, sr=sample_rate, mono=True, duration=limit_seconds)
    y = torch.from_numpy(y).float()
    window_samples = int(round(window_seconds * sample_rate))
    hop_samples = max(1, int(round(sample_rate / frame_rate_hz)))
    if y.numel() < window_samples:
        y = F.pad(y, (0, window_samples - y.numel()))
    n_windows = 1 + max(0, (y.numel() - window_samples) // hop_samples)
    starts = torch.arange(n_windows) * hop_samples
    windows = torch.stack([y[start : start + window_samples] for start in starts], dim=0)
    times = starts.numpy() / sample_rate
    return windows, times


def tensor_from_encoder_output(output, out_key="embedding_proj"):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, dict):
        for key in (out_key, "embedding", "last_hidden_state"):
            if key in output:
                return output[key]
    raise TypeError(f"Cannot extract tensor from output type {type(output)}")


def extract_track_embeddings(encoder, spec: EncoderSpec, path: str):
    windows, window_times = load_audio_windows(
        path,
        sample_rate=spec.sample_rate,
        window_seconds=spec.window_seconds,
        frame_rate_hz=FRAME_RATE_HZ,
        limit_seconds=LIMIT_SECONDS,
    )
    chunks = []
    for start in range(0, len(windows), BATCH_WINDOWS):
        batch = windows[start : start + BATCH_WINDOWS].to(DEVICE)
        if len(batch) > MAX_BATCH_WINDOWS:
            raise ValueError("BATCH_WINDOWS cannot exceed MAX_BATCH_WINDOWS in this notebook helper")
        output = encoder.extract_features(batch)
        features = tensor_from_encoder_output(output).detach().cpu()
        chunks.append(features)
    features = torch.cat(chunks, dim=0)

    if features.ndim == 3:
        # Native sequence output. Flatten windows and approximate internal frame times inside each window.
        n_windows, inner_frames, dim = features.shape
        offsets = np.linspace(0.0, spec.window_seconds, inner_frames, endpoint=False)
        times = (window_times[:, None] + offsets[None, :]).reshape(-1)
        features = features.reshape(n_windows * inner_frames, dim)
    elif features.ndim == 2:
        times = window_times
    else:
        raise ValueError(f"Unexpected feature shape: {tuple(features.shape)}")

    return {
        "path": str(path),
        "name": Path(path).stem,
        "features": features.float(),
        "times": np.asarray(times, dtype=np.float32),
    }


def downsample_for_display(features: torch.Tensor, times: np.ndarray, max_frames: int):
    if len(features) <= max_frames:
        return features, times
    idx = np.linspace(0, len(features) - 1, max_frames).round().astype(int)
    return features[idx], times[idx]


## 3. Extract Embeddings

In [ ]:
encoder, spec = instantiate_encoder(ENCODER_NAME, pool=POOL_WITHIN_WINDOW, device=DEVICE)
print(f"Loaded {ENCODER_NAME}: sample_rate={spec.sample_rate}, window={spec.window_seconds}s, pool={POOL_WITHIN_WINDOW}")
print(f"Requested frame rate: {FRAME_RATE_HZ} Hz -> hop={1.0 / FRAME_RATE_HZ:.3f}s")

tracks = []
for audio_path in AUDIO_PATHS:
    track = extract_track_embeddings(encoder, spec, audio_path)
    tracks.append(track)
    print(f"{track['name']}: features={tuple(track['features'].shape)}, duration~{track['times'][-1]:.1f}s")


## 4. Self-Similarity Matrix With MSPF Underneath

MSPF is the Music Semantic Progress Function implemented in `dyno.evaluation.temporal.compute_mspf`.

In [ ]:
def plot_ssm_and_mspf(tracks):
    n = len(tracks)
    fig, axes = plt.subplots(
        2,
        n,
        figsize=(FIGSIZE_PER_TRACK * n, 6.4),
        squeeze=False,
        gridspec_kw={"height_ratios": [4, 1]},
        constrained_layout=True,
    )
    for col, track in enumerate(tracks):
        z, t = downsample_for_display(track["features"], track["times"], MAX_FRAMES_FOR_SSM)
        ssm = compute_ssm(z)
        mspf = compute_mspf(
            z,
            window=min(MSPF_WINDOW_FRAMES, max(1, len(z) - 1)),
            sigma=MSPF_SIGMA,
            lam=MSPF_LAMBDA,
        )
        extent = [float(t[0]), float(t[-1]), float(t[-1]), float(t[0])]
        im = axes[0, col].imshow(ssm, cmap="magma", vmin=-1, vmax=1, aspect="auto", extent=extent)
        axes[0, col].set_title(track["name"])
        axes[0, col].set_xlabel("Time (s)")
        axes[0, col].set_ylabel("Time (s)")
        axes[1, col].plot(t, mspf, color="black", linewidth=1.8)
        axes[1, col].set_xlabel("Time (s)")
        axes[1, col].set_ylabel("MSPF")
    fig.colorbar(im, ax=axes[0, :], shrink=0.75, label="cosine similarity")
    return fig

plot_ssm_and_mspf(tracks)
plt.show()


## 5. PCA Component Evolution Through Time

PCA is fitted once across all provided tracks so component axes are comparable between columns.

In [ ]:
all_features = torch.cat([track["features"] for track in tracks], dim=0).numpy()
scaler = StandardScaler()
all_scaled = scaler.fit_transform(all_features)
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)
all_pca = pca.fit_transform(all_scaled)

cursor = 0
for track in tracks:
    n = len(track["features"])
    track["pca"] = all_pca[cursor : cursor + n]
    cursor += n

print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))

fig, axes = plt.subplots(
    N_PCA_COMPONENTS,
    len(tracks),
    figsize=(FIGSIZE_PER_TRACK * len(tracks), 2.2 * N_PCA_COMPONENTS),
    squeeze=False,
    sharex="col",
    constrained_layout=True,
)
for col, track in enumerate(tracks):
    for comp in range(N_PCA_COMPONENTS):
        ax = axes[comp, col]
        ax.plot(track["times"], track["pca"][:, comp], linewidth=1.2)
        ax.set_ylabel(f"PC{comp + 1}")
        if comp == 0:
            ax.set_title(track["name"])
        if comp == N_PCA_COMPONENTS - 1:
            ax.set_xlabel("Time (s)")
plt.show()


## 6. Cluster Belonging Timeline

Set `N_CLUSTERS` above, then fit K-means across all frames from all tracks. Each frame is colored by cluster index on a timeline. You can swap `KMeans` for another estimator as long as it exposes `fit_predict`.

In [ ]:
clusterer = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init="auto")
cluster_labels = clusterer.fit_predict(all_scaled)

cursor = 0
for track in tracks:
    n = len(track["features"])
    track["clusters"] = cluster_labels[cursor : cursor + n]
    cursor += n

fig, axes = plt.subplots(
    len(tracks),
    1,
    figsize=(max(8, FIGSIZE_PER_TRACK * len(tracks)), 1.25 * len(tracks)),
    squeeze=False,
    constrained_layout=True,
)
cmap = plt.get_cmap("tab20", N_CLUSTERS)
for row, track in enumerate(tracks):
    ax = axes[row, 0]
    labels = track["clusters"][None, :]
    extent = [float(track["times"][0]), float(track["times"][-1]), 0, 1]
    im = ax.imshow(labels, aspect="auto", interpolation="nearest", cmap=cmap, vmin=-0.5, vmax=N_CLUSTERS - 0.5, extent=extent)
    ax.set_yticks([])
    ax.set_ylabel(track["name"], rotation=0, ha="right", va="center")
    ax.set_xlabel("Time (s)")
fig.colorbar(im, ax=axes[:, 0], ticks=range(N_CLUSTERS), label="cluster")
plt.show()
